In [8]:
from datasets import load_dataset
import random
from tqdm import tqdm
import os
import json
import hashlib

In [9]:
def text_to_id(text):
    """Return a deterministic ID for the given text."""
    # Normalize whitespace etc. to avoid accidental differences
    return hashlib.md5(text.encode("utf-8")).hexdigest()

In [10]:
# ds = load_dataset("hotpotqa/hotpot_qa", "distractor")
ds = load_dataset("hotpotqa/hotpot_qa", "distractor")

In [11]:
num_bad_docs = 2

In [ ]:
random.seed(42) 

skip_count = 0
train_data = []
for row in tqdm(ds['train'], total=len(ds['train'])):
    query = row['question']
    good_docs = row['supporting_facts']
    good_doc_titles = good_docs["title"]
    all_docs = row['context']
    all_doc_titles = all_docs["title"]
    # print(len(all_doc_titles), len(good_doc_titles))
    good_doc_ids = [i for i, title in enumerate(all_doc_titles) if title in good_doc_titles]
    if len(all_doc_titles) >= len(good_doc_ids) + num_bad_docs:
        good_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in good_doc_ids]
        for gd in good_doc_texts:
            bad_doc_ids = random.choices([i for i in range(len(all_doc_titles)) if i not in good_doc_ids], k=num_bad_docs)
            bad_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in bad_doc_ids]
            for bd in bad_doc_texts:
                train_data.append({
                    "query": query,
                    "good_doc": gd,
                    "bad_doc": bd
                })
    else:
        skip_count += 1
    # print(train_data)
    # break
print(f"Skipped {skip_count} samples due to insufficient bad documents.")

100%|██████████| 90447/90447 [00:11<00:00, 7672.38it/s]

Skipped 418 samples due to insufficient bad documents.


In [13]:
random.seed(42) 

skip_count = 0
validation_data = []
for row in tqdm(ds['validation'], total=len(ds['validation'])):
    query = row['question']
    good_docs = row['supporting_facts']
    good_doc_titles = good_docs["title"]
    all_docs = row['context']
    all_doc_titles = all_docs["title"]
    # print(len(all_doc_titles), len(good_doc_titles))
    good_doc_ids = [i for i, title in enumerate(all_doc_titles) if title in good_doc_titles]
    if len(all_doc_titles) >= len(good_doc_ids) + num_bad_docs:
        good_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in good_doc_ids]
        for gd in good_doc_texts:
            bad_doc_ids = random.choices([i for i in range(len(all_doc_titles)) if i not in good_doc_ids], k=num_bad_docs)
            bad_doc_texts = [" ".join([s.strip() for s in all_docs["sentences"][i]]) for i in bad_doc_ids]
            for bd in bad_doc_texts:
                validation_data.append({
                    "query": query,
                    "good_doc": gd,
                    "bad_doc": bd
                })
    else:
        skip_count += 1
    # print(validation_data)
    # break
print(f"Skipped {skip_count} samples due to insufficient bad documents.")

100%|██████████| 7405/7405 [00:00<00:00, 8421.87it/s]

Skipped 28 samples due to insufficient bad documents.


In [14]:
len(train_data), len(validation_data)

(360116, 29508)

In [15]:
os.makedirs("data", exist_ok=True)
with open("data/train_data.jsonl", "w") as f:
    for item in train_data:
        f.write(json.dumps(item) + "\n")
with open("data/validation_data.jsonl", "w") as f:
    for item in validation_data:
        f.write(json.dumps(item) + "\n")

In [20]:
os.makedirs("data/docs", exist_ok=True)
doc_ids = set()
with open("data/docs/doc_data.jsonl", "w") as f:
    for item in train_data:
        gd = item["good_doc"]
        gd_id = text_to_id(gd)
        if gd_id not in doc_ids:
            doc_ids.add(gd_id)
            f.write(json.dumps({"id": gd_id, "contents": gd}) + "\n")
        bd = item["bad_doc"]
        bd_id = text_to_id(bd)
        if bd_id not in doc_ids:
            doc_ids.add(bd_id)
            f.write(json.dumps({"id": bd_id, "contents": bd}) + "\n")
    for item in validation_data:
        gd = item["good_doc"]
        gd_id = text_to_id(gd)
        if gd_id not in doc_ids:
            doc_ids.add(gd_id)
            f.write(json.dumps({"id": gd_id, "contents": gd}) + "\n")
        bd = item["bad_doc"]
        bd_id = text_to_id(bd)
        if bd_id not in doc_ids:
            doc_ids.add(bd_id)
            f.write(json.dumps({"id": bd_id, "contents": bd}) + "\n")